## Descarga


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")

print("Path to dataset files:", path)

100%|██████████| 1.17G/1.17G [00:11<00:00, 106MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21


##Visualizamos que archivos hay en el path

In [ ]:
import os

base_path = "/root/.cache/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21"

for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    for f in files:
        size = os.path.getsize(os.path.join(root, f)) / (1024*1024)
        print(f"{indent}  📄 {f}  ({size:.1f} MB)")

📁 21/
  📄 daily_dataset.csv  (335.5 MB)
  📄 darksky_parameters_documentation.html  (0.1 MB)
  📄 uk_bank_holidays.csv  (0.0 MB)
  📄 acorn_details.csv  (0.1 MB)
  📄 weather_hourly_darksky.csv  (1.9 MB)
  📄 weather_daily_darksky.csv  (0.3 MB)
  📄 informations_households.csv  (0.2 MB)
  📁 daily_dataset/
    📁 daily_dataset/
      📄 block_15.csv  (3.4 MB)
      📄 block_92.csv  (3.3 MB)
      📄 block_90.csv  (3.1 MB)
      📄 block_6.csv  (3.0 MB)
      📄 block_88.csv  (3.0 MB)
      📄 block_19.csv  (3.3 MB)
      📄 block_103.csv  (3.3 MB)
      📄 block_83.csv  (3.0 MB)
      📄 block_4.csv  (3.1 MB)
      📄 block_22.csv  (3.4 MB)
      📄 block_38.csv  (3.2 MB)
      📄 block_29.csv  (3.4 MB)
      📄 block_14.csv  (3.4 MB)
      📄 block_58.csv  (3.1 MB)
      📄 block_71.csv  (3.0 MB)
      📄 block_33.csv  (3.4 MB)
      📄 block_93.csv  (3.4 MB)
      📄 block_11.csv  (3.3 MB)
      📄 block_9.csv  (3.4 MB)
      📄 block_26.csv  (3.1 MB)
      📄 block_42.csv  (3.3 MB)
      📄 block_46.csv  (3.5 MB

Se tienen 3 carpetas con diferente granularidad

| Carpeta | Granularidad          | Tamaño total | Uso recomendado               |
| :------ | :-------------------- | :----------- | :--------------------------- |
| daily_dataset | Por día               | ~335 MB      | Clustering de hogares          |
| block_dataset | Por media hora (agrupado) | ~1.5 GB      | Balance calidad/peso           |
| halfhourly_dataset | Por media hora (raw)  | ~7 GB        | ⚠️ Muy pesado para Colab |



¿Por qué no halfhourly_dataset?
En Colab se tienen 12GB de RAM. Cargar los 112 bloques del halfhourly (~7GB descomprimido en memoria son fácilmente 15-20GB) va a crashear la sesión. No vale la pena

¿Por qué daily_dataset como base?
Para clustering de hogares y predicción de consumo mensual, la granularidad diaria es suficiente y manejable. Además ya existe el archivo consolidado daily_dataset.csv (335MB)

¿Por qué complementar con hhblock_dataset?
Para el análisis de patrones horarios (franja pico/valle) que enriquece el EDA y la narrativa ejecutiva. Pero solo carga una muestra de bloques, no los 112.

---
## confirmar la estructura de cada archivo:

In [ ]:
import pandas as pd

base = "/root/.cache/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21"

# Ver estructura de los archivos clave
archivos = {
    "daily_dataset.csv": f"{base}/daily_dataset.csv",
    "informations_households.csv": f"{base}/informations_households.csv",
    "acorn_details.csv": f"{base}/acorn_details.csv",
    "weather_daily_darksky.csv": f"{base}/weather_daily_darksky.csv",
    "hhblock_dataset/block_0.csv": f"{base}/hhblock_dataset/hhblock_dataset/block_0.csv"
}

for nombre, path in archivos.items():
    df_tmp = pd.read_csv(path, nrows=3, encoding='latin1')
    print(f"\n{'='*50}")
    print(f"📄 {nombre}")
    print(f"   Columnas: {df_tmp.columns.tolist()}")
    print(df_tmp.head(2))


📄 daily_dataset.csv
   Columnas: ['LCLid', 'day', 'energy_median', 'energy_mean', 'energy_max', 'energy_count', 'energy_std', 'energy_sum', 'energy_min']
       LCLid         day  energy_median  energy_mean  energy_max  \
0  MAC000131  2011-12-15         0.4850     0.432045       0.868   
1  MAC000131  2011-12-16         0.1415     0.296167       1.116   

   energy_count  energy_std  energy_sum  energy_min  
0            22    0.239146       9.505       0.072  
1            48    0.281471      14.216       0.031  

📄 informations_households.csv
   Columnas: ['LCLid', 'stdorToU', 'Acorn', 'Acorn_grouped', 'file']
       LCLid stdorToU   Acorn Acorn_grouped     file
0  MAC005492      ToU  ACORN-        ACORN-  block_0
1  MAC001074      ToU  ACORN-        ACORN-  block_0

📄 acorn_details.csv
   Columnas: ['MAIN CATEGORIES', 'CATEGORIES', 'REFERENCE', 'ACORN-A', 'ACORN-B', 'ACORN-C', 'ACORN-D', 'ACORN-E', 'ACORN-F', 'ACORN-G', 'ACORN-H', 'ACORN-I', 'ACORN-J', 'ACORN-K', 'ACORN-L', 'ACORN




| Archivo | Qué aporta al proyecto |
| :---------------------------- | :-------------------------------------------------------------------------------------- |
| daily_dataset.csv             | Base principal — consumo diario por hogar                                               |
| informations_households.csv   | Tipo de tarifa (ToU vs Std) + grupo Acorn socioeconómico                                |
| weather_daily_darksky.csv     | Temperatura, humedad, nubosidad — variables externas clave                                |
| hhblock_dataset/block_0.csv   | 48 columnas = consumo cada 30 min → perfil diario por hogar                             |
| acorn_details.csv             | Contexto demográfico por grupo (no se modela, se usa para interpretar)                   |
---


Una observación importante: el hhblock_dataset tiene los 48 periodos del día como columnas (hh_0 a hh_47), no como filas. Eso requiere un `melt()` para dejarlo utilizable.
